# 情報数学Ⅲ　第3回

In [ ]:
# 必要なデータファイルを取得
import requests

base_url = "https://raw.githubusercontent.com/logics-of-blue/book-python-stats-2nd/refs/heads/main/book-data/"
filenames = [
    "3-6-1-fish_multi.csv"
]

for filename in filenames:
    url = base_url + filename
    print(f"Downloading {filename}...")
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, "wb") as f:
            f.write(response.content)
    else:
        print(f"Failed to download {filename}: {response.status_code}")


# 第3部　記述統計

## 6章　層別分析

### 実装：分析の準備

In [ ]:
# 数値計算に使うライブラリ
import numpy as np
import pandas as pd

# 複雑な統計処理を行うライブラリ
from scipy import stats

# グラフを描画するライブラリ
from matplotlib import pyplot as plt
import seaborn as sns
sns.set()

In [ ]:
# 表示設定(書籍本文のレイアウトと合わせるためであり、必須ではありません)
np.set_printoptions(linewidth=60)
pd.set_option('display.width', 60)

### 実装：分析対象となるデータの用意

In [ ]:
fish_multi = pd.read_csv('3-6-1-fish_multi.csv')
print(fish_multi.head(3))

In [ ]:
print(fish_multi)

In [ ]:
# サンプルサイズ
len(fish_multi)

In [ ]:
# 魚の種類
fish_multi['species'].value_counts()

In [ ]:
# 標本平均
np.mean(fish_multi['length'])

### 実装：グループ別の統計量の計算

#### グループ別の平均値

In [ ]:
# 魚の種類ごとの集計
group = fish_multi.groupby('species')
print(group.mean())

In [ ]:
# 1行でまとめて記載
print(fish_multi.groupby('species').mean())

#### グループ別の要約統計量

In [ ]:
# 要約統計量
print(group.describe())

#### pandas以外の関数を使う

In [ ]:
print(group.agg(stats.mode))

### 実装：ペンギンデータの読み込み

#### データの読み込み

In [ ]:
# seaborn組み込みのペンギンのデータを取得
penguins = sns.load_dataset('penguins')
print(penguins.head(n=2))

In [ ]:
penguins


#### データのチェック

In [ ]:
# 鳥の種類の分布
penguins['species'].value_counts()

In [ ]:
# Torgersen島のデータだけを抽出
penguins.query('island == "Torgersen"')['species'].value_counts()

In [ ]:
# Biscoe島のデータだけを抽出
penguins.query('island == "Biscoe"')['species'].value_counts()

In [ ]:
# Torgersen島のデータだけを抽出
penguins.query('island == "Dream"')['species'].value_counts()

### 実装：ペンギンデータの層別分析

In [ ]:
# ベンギンの種別・性別の集計
group_penguins = penguins[['species', 'sex', 'body_mass_g']].groupby(['species', 'sex'])
# group_penguins = penguins.groupby(['species', 'sex']) # 鈴木による修正．オリジナル→古いpandasでは通る．現在は集計する数値を明示する必要あり
print(group_penguins.mean()['body_mass_g'])

In [ ]:
# ベンギンの種別・島別・性別の集計
group_penguins = penguins.groupby(['species', 'island', 'sex'])
print(group_penguins.mean()['body_mass_g'])

### 実装：欠損値の扱いに注意

#### 欠測値

In [ ]:
# body_mass_gの４番目の値は欠測
print(penguins[['species','body_mass_g']].head(n = 4))

In [ ]:
# 参考：欠損データの抽出（教科書には載っていないコードです）
# Adelie種のペンギンでbody_mass_gが欠測であるのは１データのみです
penguins[penguins.isnull().any(axis=1)]

#### 欠測値に対する挙動

In [ ]:
# 種別でグループ分けしてbody_mass_gのサンプルサイズを調べる
group_sp = penguins.groupby(['species'])
print(group_sp.count()['body_mass_g'])

In [ ]:
# サンプルサイズを151と考えて平均値を計算
round(group_sp.sum()['body_mass_g'].Adelie / 151, 3)

In [ ]:
# 層別に平均値を計算
# round(group_sp.mean()['body_mass_g'].Adelie, 3) # 鈴木による修正．古いpandasなら通るが，現在は数値列を勝手に選んでくれない
round(group_sp['body_mass_g'].mean().loc['Adelie'], 3)

### 実装：単純なヒストグラム

In [ ]:
bins = np.arange(2,11,1)
bins

In [ ]:
sns.histplot(x='length',      # x軸
             data=fish_multi, # データ
             bins=bins,       # bins
             color='gray')  # 色の指定(グレースケール)

### 実装：グループ別のヒストグラム

In [ ]:
sns.histplot(x='length',      # x軸
             hue='species',   # 色分けの対象
             data=fish_multi, # データ
             bins=bins,       # bins
             palette='gray')  # 色の指定(グレースケール)

#### （以下は参考）層別のカーネル密度推定も、ヒストグラムと同様に実行可能です

In [ ]:
# 単純なカーネル密度推定の結果(書籍には載っていないコードです)
sns.kdeplot(data=fish_multi, # データ
             x='length',     # x軸
             color='gray')   # 色の指定(グレースケール)

In [ ]:
# グループ別にしたカーネル密度推定の結果(書籍には載っていないコードです)
sns.kdeplot(data=fish_multi,  # データ
             x='length',      # x軸
             hue='species',   # 色分けの対象
             palette='gray')  # 色の指定(グレースケール)